# Calls Data Cleaning

This notebook prepares CRM call records for downstream sales and funnel analysis.

### Main tasks
- standardize column names and reconnect duplicated contact IDs
- remove empty or analytically redundant fields
- parse call timestamps
- remove duplicate call records
- inspect missing contact links and consistency issues
- create an analytical `is_successful` call flag

> **Data note:** the original CRM files are not included in the public repository.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from helpers import to_snake, df_overview, df_clean_summary

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load and Initial Inspection

In [ ]:
calls = pd.read_excel(
    RAW_DIR / 'Calls (Done).xlsx',
    dtype={'Id': str, 'CONTACTID': str}
)

In [ ]:
# Standardize column names to snake_case
calls.columns = [to_snake(c) for c in calls.columns]

In [ ]:
# Reconnect calls linked to duplicate contacts to the retained contact ID
id_replacements = pd.read_pickle(
    PROCESSED_DIR / 'contacts_id_replacements.pkl'
)
calls['contactid'] = calls['contactid'].replace(id_replacements)

In [ ]:
# Display a compact data-quality overview
df_overview(calls)

In [ ]:
# Inspect unique values in categorical fields
exclude_cols = ['id', 'contactid', 'call_start_time', 'call_duration']

for col in calls.columns:
    if col not in exclude_cols:
        print(f"\nUnique values in '{col}':")
        print(calls[col].unique())

## 2. Remove Empty and Analytically Redundant Columns

In [ ]:
# Check exact duplicate rows
calls.duplicated().sum()

In [ ]:
# Check whether these fields are populated only for Outbound calls

In [ ]:
pd.crosstab(calls['call_type'], calls['outgoing_call_status'].isna(), margins=True, margins_name='Total')

In [ ]:
pd.crosstab(calls['call_type'], calls['scheduled_in_crm'].isna(), margins=True, margins_name='Total')

- `dialled_number` and `tag` are completely empty and are removed.
- `outgoing_call_status` and `scheduled_in_crm` are populated only for Outbound calls and do not add useful analytical information for this project, so they are removed.
- `contactid` contains missing values for calls that are not linked to a CRM contact.
- Missing `call_duration` values are retained as `NaN`; pandas will ignore them in relevant summary statistics.

In [ ]:
cols_to_drop = [
    'dialled_number',
    'tag',
    'outgoing_call_status',
    'scheduled_in_crm'
]
calls = calls.drop(columns=cols_to_drop)

print(f'Removed columns: {cols_to_drop}')
print(f'Remaining columns: {calls.shape[1]}')

## 3. Data Types

In [ ]:
# Parse call timestamp
calls['call_start_time'] = pd.to_datetime(
    calls['call_start_time'],
    format='%d.%m.%Y %H:%M',
    errors='coerce'
)

In [ ]:
print('Data types after conversion:')
print(calls.dtypes)
print(f'\nCalls without contactid: {calls["contactid"].isna().sum()}')

## 4. Duplicate Calls

In [ ]:
# Duplicate definition used in this project:
# same timestamp + manager + contact + duration.
# Such duplicates can occur when CRM creates repeated call records.
dup_cols = [
    'call_start_time',
    'call_owner_name',
    'contactid',
    'call_duration'
]

n_dups_total = calls.duplicated(subset=dup_cols, keep=False).sum()
n_dups_to_drop = calls.duplicated(subset=dup_cols, keep='first').sum()

print(f'Rows belonging to duplicate groups: {n_dups_total}')
print(f'Duplicate copies to remove: {n_dups_to_drop}')

calls = calls.drop_duplicates(subset=dup_cols, keep='first')
print(f'Rows after deduplication: {len(calls)}')

## 5. Missing Contact Links

In [ ]:
# Inspect call types among records without a linked contact
calls.loc[calls['contactid'].isna(), 'call_type'].value_counts()

In [ ]:
# No clear pattern was identified, so these missing links are retained.

## 6. Successful Call Flag

In [ ]:
calls['call_status'].unique()

In [ ]:
successful_statuses = ['Attended Dialled', 'Received', 'Scheduled Attended', 'Scheduled Attended Delay']

calls[calls['call_status'].isin(successful_statuses)]['call_duration'].describe()

In [ ]:
# Analytical assumption used in the project:
# a successful call has a successful CRM status and lasts more than 30 seconds.
min_duration = 30

calls['is_successful'] = (
    calls['call_status'].isin(successful_statuses)
    & (calls['call_duration'] > min_duration)
)

successful_calls = calls['is_successful'].sum()

print(
    f'is_successful: {successful_calls} of {len(calls)} calls '
    f'({successful_calls / len(calls) * 100:.2f}%)'
)

In [ ]:
# Check successful calls among records without a linked contact
print(
    calls.loc[calls['contactid'].isna(), 'is_successful']
    .value_counts(normalize=True)
)
print()
print(f"Calls without contactid: {calls['contactid'].isna().sum()}")
print(
    "Successful calls without contactid: "
    f"{calls.loc[calls['contactid'].isna(), 'is_successful'].sum()}"
)

# Project interpretation: these may represent calls that were recorded
# without being linked to the corresponding CRM contact.

## 7. Data Consistency Checks

### 7.1. `call_type` vs `call_status`

In [ ]:
pd.crosstab(calls['call_type'], calls['call_status'])

One record is classified as both `Inbound` and `Missed`. Under the CRM interpretation used in this project, this combination is treated as inconsistent because `Missed` is represented as a separate call type.

In [ ]:
calls[(calls['call_type'] == 'Inbound') & (calls['call_status'] == 'Missed')]

This single record is treated as a likely CRM entry inconsistency and is not material to the overall analysis.

### 7.2. Calls During Night Hours

In [ ]:
calls[calls['call_start_time'].dt.hour < 7]

In [ ]:
len(calls[calls['call_start_time'].dt.hour < 7])

The project identified 24 calls between 00:00 and 07:00. All were `Missed` with zero duration, so they do not materially affect call-performance metrics.

## 8. Final Quality Check

In [ ]:
df_clean_summary(calls)

## 9. Save Processed Data

In [ ]:
calls.to_pickle(PROCESSED_DIR / 'calls_clean.pkl')

print('Saved: calls_clean.pkl')
print(f'Shape: {calls.shape}')

## 10. Output Dataset

| Column | Type | Description |
|---|---|---|
| `id` | `object` | Unique call ID |
| `call_start_time` | `datetime64` | Call start timestamp |
| `call_owner_name` | `object` | Manager who handled the call |
| `contactid` | `object` | Linked CRM contact ID |
| `call_type` | `object` | Inbound / Outbound / Missed |
| `call_duration` | `float64` | Call duration in seconds |
| `call_status` | `object` | CRM call outcome |
| `is_successful` | `bool` | Project-defined success flag: successful status and duration > 30 sec |

**Dataset relationship**
- `calls.contactid` ↔ `contacts.id`

## 11. Cleaning Summary

**Source table:** CRM Calls with fields describing call timing, ownership, contact linkage, call type, duration, status, and CRM metadata.

| Step | Action | Result |
|---|---|---|
| 1 | Column standardization | Converted column names to `snake_case` |
| 2 | Empty-field removal | Removed completely empty fields such as `dialled_number` and `tag` |
| 3 | Redundant-field removal | Removed fields used only for Outbound calls when they did not add analytical value |
| 4 | Missing-value handling | Retained missing `call_duration` and missing contact links where no justified imputation was available |
| 5 | Datetime conversion | Converted `call_start_time` to `datetime64` |
| 6 | Deduplication | Removed repeated calls based on timestamp + manager + contact + duration |
| 7 | Analytical success flag | Added `is_successful` based on successful CRM statuses and a >30-second duration threshold |

### Output

- `calls_clean.pkl` — cleaned Calls dataset with the `is_successful` analytical flag